<a href="https://colab.research.google.com/github/EMADUDDINAsdaq/federated-learning-fairness-xray/blob/main/dataset_freezing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dataset Freezing Notebook — CSC8639
Emaduddin Asdaq Syed Mohammed | MSc Data Science and AI

Freezes the Dirichlet partition and patient-disjoint 85/10/5 split into CSV files
so all five FL method notebooks (FedAvg, q-FedAvg, Ada-IFFL, GIFAIR-Global, GIFAIR-Per)
train and evaluate on the exact same data.

. Nothing beyond this notebook needs to change in the five model notebooks
except the cell that loads train_clients / val_clients / test_clients.

## Cell 1 — Environment Setup

In [ ]:
!pip install "flwr[simulation]" protobuf -q
print("✓ Libraries installed")

import os, json, time, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.metrics import roc_auc_score
warnings.filterwarnings('ignore')

print(f"torch : {torch.__version__}")
print(f"numpy : {np.__version__}")
print(f"GPU   : {torch.cuda.is_available()}")

## Cell 2 — Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Cell 3 — Dataset Download (Kaggle)

In [ ]:
import shutil

os.makedirs('/root/.kaggle', exist_ok=True)
shutil.copy('/content/drive/MyDrive/dissertation/kaggle.json',
            '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)

os.system('pip install -q kaggle')
os.system('kaggle datasets download -d nih-chest-xrays/data '
          '--path /content/nih_kaggle --unzip --quiet')

DATASET_PATH = '/content/nih_kaggle'
print(f"✓ Dataset path: {DATASET_PATH}")

## Cell 4 — Image Indexing

In [ ]:
image_path_dict = {}
for folder in sorted(os.listdir(DATASET_PATH)):
    if folder.startswith('images_'):
        img_dir = os.path.join(DATASET_PATH, folder, 'images')
        if os.path.isdir(img_dir):
            for f in os.listdir(img_dir):
                if f.endswith('.png'):
                    image_path_dict[f] = os.path.join(img_dir, f)
print(f"✓ Indexed {len(image_path_dict):,} images")

## Cell 5 — Metadata Loading and Cleaning

In [ ]:
df_raw = pd.read_csv(f"{DATASET_PATH}/Data_Entry_2017.csv")

df = df_raw.copy()
df = df.rename(columns={'Patient Gender': 'Patient Sex'})

null_cols = [c for c in df.columns if df[c].isnull().all()]
df = df.drop(columns=null_cols)

df = df[df['Patient Age'] <= 100]

df['label'] = (df['Finding Labels'] != 'No Finding').astype(int)

df['Age Group'] = pd.cut(
    df['Patient Age'],
    bins=[0, 20, 40, 60, 80, 101],
    labels=['0-20', '20-40', '40-60', '60-80', '80+']
)

df['image_path'] = df['Image Index'].map(image_path_dict)
df = df.dropna(subset=['image_path']).reset_index(drop=True)

print(f"✓ Final dataset : {len(df):,} rows")
print(f"  Pathology rate: {df['label'].mean()*100:.1f}%")

## Cell 6 — Dirichlet Partition (identical to all 5 training notebooks)

In [ ]:
np.random.seed(42)

NUM_CLIENTS    = 5
ALPHA          = 0.5
HOSPITAL_NAMES = ['Hospital_A', 'Hospital_B', 'Hospital_C',
                  'Hospital_D', 'Hospital_E']

idx_0 = np.where(df['label'].values == 0)[0]
idx_1 = np.where(df['label'].values == 1)[0]
np.random.shuffle(idx_0)
np.random.shuffle(idx_1)

props_0 = np.random.dirichlet(np.repeat(ALPHA, NUM_CLIENTS))
props_1 = np.random.dirichlet(np.repeat(ALPHA, NUM_CLIENTS))

splits_0 = (props_0 * len(idx_0)).astype(int)
splits_1 = (props_1 * len(idx_1)).astype(int)
splits_0[-1] = len(idx_0) - splits_0[:-1].sum()
splits_1[-1] = len(idx_1) - splits_1[:-1].sum()

clients = {}
ptr0, ptr1 = 0, 0
for i, name in enumerate(HOSPITAL_NAMES):
    idx = np.concatenate([
        idx_0[ptr0:ptr0 + splits_0[i]],
        idx_1[ptr1:ptr1 + splits_1[i]]
    ])
    clients[name] = df.iloc[idx].copy().reset_index(drop=True)
    ptr0 += splits_0[i]
    ptr1 += splits_1[i]

for name, data in clients.items():
    print(f"{name}: {len(data):,} images | pathology {data['label'].mean()*100:.1f}%")
print("\n✓ Non-IID partition confirmed")

## Cell 7 — Train/Val/Test Split (patient-disjoint, 85/10/5)

In [ ]:
# Same 85/10/5 ratios already used in all five notebooks, but grouped by
# Patient ID so the same patient's scans never appear in both train and test.
# Variable names (train_clients / val_clients / test_clients) are unchanged
# so nothing downstream in any model notebook needs to be edited.

TRAIN_SPLIT = 0.85
VAL_SPLIT   = 0.10
# remaining 0.05 = test

train_clients = {}
val_clients   = {}
test_clients  = {}

for name, data in clients.items():
    patient_ids = data['Patient ID'].unique()
    rng = np.random.RandomState(42)
    rng.shuffle(patient_ids)

    n       = len(patient_ids)
    n_train = int(n * TRAIN_SPLIT)
    n_val   = int(n * VAL_SPLIT)

    train_ids = set(patient_ids[:n_train])
    val_ids   = set(patient_ids[n_train:n_train + n_val])
    test_ids  = set(patient_ids[n_train + n_val:])

    train_clients[name] = data[data['Patient ID'].isin(train_ids)].reset_index(drop=True)
    val_clients[name]   = data[data['Patient ID'].isin(val_ids)].reset_index(drop=True)
    test_clients[name]  = data[data['Patient ID'].isin(test_ids)].reset_index(drop=True)

print(f"{'Client':<14} {'Train':>8} {'Val':>6} {'Test':>6}")
for name in HOSPITAL_NAMES:
    print(f"{name:<14} {len(train_clients[name]):>8,} "
          f"{len(val_clients[name]):>6,} {len(test_clients[name]):>6,}")

## Cell 8 — Sanity Check: Confirm No Patient Overlap

In [ ]:
for name in HOSPITAL_NAMES:
    tr = set(train_clients[name]['Patient ID'])
    vl = set(val_clients[name]['Patient ID'])
    te = set(test_clients[name]['Patient ID'])
    overlap = len((tr & vl) | (tr & te) | (vl & te))
    print(f"{name}: patient overlap = {overlap}")

## Cell 9 — Save Frozen CSVs to Drive

In [ ]:
SPLIT_DIR = '/content/drive/MyDrive/dissertation/splits'
os.makedirs(SPLIT_DIR, exist_ok=True)

for name in HOSPITAL_NAMES:
    train_clients[name].to_csv(f'{SPLIT_DIR}/{name}_train.csv', index=False)
    val_clients[name].to_csv(f'{SPLIT_DIR}/{name}_val.csv', index=False)
    test_clients[name].to_csv(f'{SPLIT_DIR}/{name}_test.csv', index=False)

print("✓ 15 CSVs saved to:", SPLIT_DIR)

## Loading in the five training notebooks

Replace the split-generation part of Section 5/6 in each of the five model notebooks
(FedAvg, q-FedAvg, Ada-IFFL, GIFAIR-Global, GIFAIR-Per) with this single cell.
Everything else in those notebooks (ChestXrayDataset, build_model, evaluate_client,
client classes, strategies, training loop) stays exactly as it is.

```python
SPLIT_DIR = '/content/drive/MyDrive/dissertation/splits'
HOSPITAL_NAMES = ['Hospital_A', 'Hospital_B', 'Hospital_C',
                  'Hospital_D', 'Hospital_E']

train_clients = {n: pd.read_csv(f'{SPLIT_DIR}/{n}_train.csv') for n in HOSPITAL_NAMES}
val_clients   = {n: pd.read_csv(f'{SPLIT_DIR}/{n}_val.csv')   for n in HOSPITAL_NAMES}
test_clients  = {n: pd.read_csv(f'{SPLIT_DIR}/{n}_test.csv')  for n in HOSPITAL_NAMES}

for name in HOSPITAL_NAMES:
    print(f"{name}: {len(train_clients[name]):,} train / "
          f"{len(val_clients[name]):,} val / {len(test_clients[name]):,} test")
```